# Local Common Voice Sample Generator

This notebook processes locally downloaded Common Voice datasets from `/Desktop/cv/` and generates staged game entries.

## Workflow
1. Download CV datasets manually for each language to `~/Desktop/cv/{lang_code}/`
2. Run this notebook to extract N samples from each language
3. Samples are saved to `staging.csv` with all required game fields
4. Shuffle `staging.csv` and append rows to `game_data.csv` as needed
5. Delete the downloaded CV folders to save space

## Configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import subprocess
import shutil
import soundfile as sf
import random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
import time
import os
from datetime import date, timedelta
import pprint

# Paths
CV_BASE_DIR = Path.home() / 'Desktop' / 'cv' 
LANGUAGE_DATA_PATH = 'languages.csv'
STAGING_CSV_PATH = 'staging.csv'
GAME_DATA_PATH = 'game_data.csv'
AUDIO_DIR = Path('assets/audio')
AUDIO_DIR.mkdir(exist_ok=True)

# How many samples to extract from each language
SAMPLES_PER_LANGUAGE = 60

# Minimum audio length (in samples) to consider valid
MIN_AUDIO_LENGTH = 200000 

# Missing translation marker
MISSING_TRANSLATION_TOKEN = "???????"

print(f"CV Base Directory: {CV_BASE_DIR}")
print(f"CV Base Directory exists: {CV_BASE_DIR.exists()}")

CV Base Directory: /Users/raymondtana/Desktop/cv
CV Base Directory exists: True


## Helper Functions (copied from batch_generator.ipynb)

In [2]:
def find_espeak():
    """Return path to espeak-ng or espeak binary, raise if missing."""
    exe = shutil.which("espeak-ng") or shutil.which("espeak")
    if exe is None:
        raise FileNotFoundError("No espeak-ng/espeak binary on PATH")
    return exe

def phonemize(text: str, lang: str = "en", ipa: bool = False,
              ipa_level: int = 1, keep_stress: bool = True) -> str:
    """Phonemize text using eSpeak (NG)"""
    try: 
        exe = find_espeak()

        # Build the command line
        args = [exe, "-q", f"-v{lang}"]
        if ipa:
            args.append(f"--ipa={ipa_level}")
        else:
            args.append("-x")
            if not keep_stress:
                args.append("--sep=-")
        args.append(text)

        # Run the command
        proc = subprocess.run(args, text=True, capture_output=True, check=True)
        out = proc.stdout.strip()

        if not keep_stress:
            out = out.replace("ˈ", "")

        # Remove line breaks and underscores
        out = out.replace("\n", " ").replace("_", "").replace("\r", " ").strip()

        return out
    
    except Exception as e:
        print(f"Error getting IPA representation: {str(e)}")
        return "???"

# Exploit Google Translate to translate this sentence (webscraping!)
def translate(language, text):
    
    options = webdriver.ChromeOptions()
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    driver = webdriver.Chrome(options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    try:
        # Navigate to Google Translate
        driver.get("https://translate.google.com")
        wait = WebDriverWait(driver, 15)
        actions = ActionChains(driver)
        
        # Wait for page to load
        time.sleep(0.5)
        
        # Enter text (cursor should be ready in text area)
        actions.send_keys(text).perform()
        
        # Find and click the target language button
        target_lang_button = wait.until(EC.element_to_be_clickable((
            By.CSS_SELECTOR, 
            "button[aria-label*='target language'], .VfPpkd-Bz112c-RLmnJb"
        )))
        target_lang_button.click()
        
        # Wait briefly for dropdown, then type language and press Enter
        time.sleep(0.5)
        actions.send_keys(language).perform()
        time.sleep(0.5)
        actions.send_keys(Keys.RETURN).perform()
        
        # Wait for translation and extract
        time.sleep(0.5)
        
        # Extract translation
        translation_selectors = [
            "span[jsname='W297wb']",
            "span[lang]:not([lang='auto']):not([lang=''])",
            ".ryNqvb span",
            ".J0lOec span"
        ]
        
        for selector in translation_selectors:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                for element in elements:
                    translated_text = element.text.strip()
                    if translated_text and translated_text != text:
                        print(f"Translation found: {translated_text}")
                        return translated_text
            except:
                continue
        return None
        
    except Exception as e:
        print(f"Error: {str(e)}")
        return None
        
    finally:
        driver.quit()

## Scan CV Directory for Available Languages

In [3]:
def scan_cv_directory(cv_base_dir):
    """
    Scan the CV base directory for language folders.
    Each folder should contain:
    - clips/ (directory with .mp3 files)
    - validated.tsv (metadata file) 
        features: {client_id, path, sentence_id, sentence, sentence_domain, up_votes, down_votes, age, gender, accents, variant, locale, segment}
    
    Returns: dict mapping cv_code -> folder_path
    """
    if not cv_base_dir.exists():
        print(f"❌ CV directory doesn't exist: {cv_base_dir}")
        return {}
    
    available_langs = {}
    
    for folder in cv_base_dir.iterdir():
        if not folder.is_dir():
            continue
        
        clips_dir = folder / 'clips'
        validated_tsv = folder / 'validated.tsv'
        
        # Check if this folder has the expected structure
        if clips_dir.exists() and validated_tsv.exists():
            cv_code = folder.name
            available_langs[cv_code] = folder
            print(f"✓ Found {cv_code} at {folder}")
        else:
            print(f"⚠ Skipping {folder.name} (missing clips/ or validated.tsv)")
    
    return available_langs

# Scan for available languages
available_cv_langs = scan_cv_directory(CV_BASE_DIR)
print(f"\n📊 Found {len(available_cv_langs)} language(s) ready to process")

✓ Found de at /Users/raymondtana/Desktop/cv/de

📊 Found 1 language(s) ready to process


## Load Language Metadata

In [4]:
# Load your languages.csv metadata
languages_df = pd.read_csv(LANGUAGE_DATA_PATH)

# Create mapping from cv_code to language metadata
cv_to_meta = {}
for _, row in languages_df.iterrows():
    cv_code = row['cv_code']
    cv_to_meta[cv_code] = {
        'english_name': row['english_name'],
        'iso3': row['iso3'],
        'espeak_code': row['espeak_code'],
        'lineage': eval(row['lineage']) if isinstance(row['lineage'], str) else row['lineage'],
        'family_0': row['family_0'],
        'family_1': row['family_1'],
        'family_2': row['family_2']
    }

print(f"Loaded metadata for {len(cv_to_meta)} languages")

available_codes = list(available_cv_langs.keys())
print(f"loaded languages have metadata?")
pprint.pprint(list(zip(available_codes, [code in cv_to_meta for code in available_codes])))

Loaded metadata for 46 languages
loaded languages have metadata?
[('de', True)]


## Extract Samples from Local CV Dataset

In [5]:
def extract_samples_from_local_cv(cv_code, cv_folder, language_meta, num_samples=30):
    """
    Extract N random samples from a local CV dataset folder.
    
    Returns: list of sample dicts ready to add to staging CSV
    """
    validated_tsv = cv_folder / 'validated.tsv'
    clips_dir = cv_folder / 'clips'
    
    print(f"\n{'='*60}")
    print(f"Processing {language_meta['english_name']} ({cv_code})")
    print(f"{'='*60}")
    
    try:
        # Load validated.tsv
        df = pd.read_csv(validated_tsv, sep='\t')
        print(f"  Loaded {len(df)} validated samples")
        
        # Shuffle and prepare to sample
        df = df.sample(frac=1, random_state=random.randint(0, 2**32-1)).reset_index(drop=True)
        
        samples_collected = []
        attempted = 0
        
        for idx, row in df.iterrows():
            if len(samples_collected) >= num_samples:
                break
            
            attempted += 1
            if attempted > len(df):
                print(f"  ⚠ Ran out of samples (only found {len(samples_collected)}/{num_samples})")
                break
            
            # Get audio file path
            audio_filename = row['path']
            audio_path = clips_dir / audio_filename
            
            if not audio_path.exists():
                continue
            
            # Read audio
            try:
                wave, rate = sf.read(audio_path)
            except Exception as e:
                print(f"  ✗ Error reading {audio_filename}: {e}")
                continue
            
            # Check minimum length
            if wave.shape[0] < MIN_AUDIO_LENGTH:
                continue
            
            # Extract sentence
            sentence = str(row['sentence']).replace('\n', ' ').strip()
            if not sentence or sentence == 'nan':
                continue
            
            # Generate IPA
            ipa = phonemize(
                text=sentence,
                lang=language_meta['espeak_code'],
                ipa=True,
                ipa_level=1,
                keep_stress=False
            )
            
            # Translate to English
            translation = translate(language_meta['english_name'], sentence)
            if not translation:
                translation = MISSING_TRANSLATION_TOKEN  # Mark for manual translation
            
            # Save audio to staging area with temporary name
            temp_audio_filename = f"staging_{cv_code}_{len(samples_collected):03d}.mp3"
            temp_audio_path = AUDIO_DIR / temp_audio_filename
            sf.write(temp_audio_path, wave, rate, format='MP3')
            
            # Create sample record
            sample = {
                'date': None,  # Will be assigned when moving to game_data.csv
                'language': language_meta['english_name'],
                'iso': language_meta['iso3'],
                'cv_code': cv_code,
                'espeak_code': language_meta['espeak_code'],
                'lineage': str(language_meta['lineage']),
                'family_0': language_meta['family_0'],
                'family_1': language_meta['family_1'],
                'family_2': language_meta['family_2'],
                'sentence': sentence,
                'translation': translation,
                'wave': temp_audio_filename,
                'sampling_rate': rate,
                'IPA': ipa
            }
            
            samples_collected.append(sample)
            
            if len(samples_collected) % 5 == 0:
                print(f"  Progress: {len(samples_collected)}/{num_samples} samples")
        
        print(f"  ✓ Collected {len(samples_collected)} samples")
        return samples_collected
        
    except Exception as e:
        print(f"  ❌ Error processing {cv_code}: {e}")
        import traceback
        traceback.print_exc()
        return []

## Process All Available Languages

In [6]:
# Load existing staging CSV if it exists
if Path(STAGING_CSV_PATH).exists():
    staging_df = pd.read_csv(STAGING_CSV_PATH)
    print(f"Loaded existing staging.csv with {len(staging_df)} samples")
    
    # Track which languages have already been processed
    if 'cv_code' in staging_df.columns:
        processed_codes = set(staging_df['cv_code'].unique())
        print(f"Already processed languages: {processed_codes}")
    else:
        processed_codes = set()
else:
    staging_df = pd.DataFrame()
    processed_codes = set()
    print("Creating new staging.csv")

# Process each available language
for cv_code, cv_folder in available_cv_langs.items():
    # Skip if already processed
    if cv_code in processed_codes:
        print(f"\n⏭️  Skipping {cv_code} (already in staging.csv)")
        continue
    
    # Check if we have metadata for this language
    if cv_code not in cv_to_meta:
        print(f"\n⚠ No metadata found for {cv_code}, skipping")
        continue
    
    # Extract samples
    samples = extract_samples_from_local_cv(
        cv_code=cv_code,
        cv_folder=cv_folder,
        language_meta=cv_to_meta[cv_code],
        num_samples=SAMPLES_PER_LANGUAGE
    )
    
    # Save incrementally after each language
    if samples:
        new_samples_df = pd.DataFrame(samples)
        staging_df = pd.concat([staging_df, new_samples_df], ignore_index=True)
        
        # Save to CSV immediately
        staging_df.to_csv(STAGING_CSV_PATH, index=False)
        processed_codes.add(cv_code)
        
        print(f"  💾 Saved {len(samples)} samples to {STAGING_CSV_PATH}")
        print(f"  📊 Total samples in staging: {len(staging_df)}")

print(f"\n✅ Processing complete!")
print(f"📊 Final total: {len(staging_df)} samples from {len(processed_codes)} languages")

Loaded existing staging.csv with 2395 samples
Already processed languages: {'bg', 'ro', 'is', 'cy', 'it', 'sw', 'vi', 'ne-NP', 'nl', 'pl', 'yue', 'sr', 'fa', 'id', 'cs', 'pt', 'hy-AM', 'lv', 'et', 'pa-IN', 'da', 'hu', 'el', 'af', 'zh-CN', 'ml', 'fi', 'ga-IE', 'ka', 'es', 'ta', 'hi', 'mk', 'sq', 'fr', 'tr', 'sk', 'sv-SE', 'ru', 'lt'}

Processing German (de)
  Loaded 944924 validated samples
Translation found: The International Film Encyclopedia wrote that the film was tailored to the lead actor.
Translation found: Since Hangover, everyone knows what Rohypnol is.
Translation found: I will arrive by long-distance bus directly at the central bus station in Ulm.
Translation found: The DNS server is used to look up unknown addresses.
Translation found: They could have had the age-old spectacle of a national interests bazaar even without a convention.
  Progress: 5/60 samples
Translation found: Those affected often suffer from the traumatic experiences for decades.
Translation found: The worl

### Fixing Any Missing Translations

Sometimes the Google Translate site blocks translations requests when we've inundated it too much from the same IP address in too short a time interval. This block runs through the `staging.csv` file to try and replace any missing translations using the same `translate(language_meta['english_name'], sentence)` call. Missing values are always shown as `MISSING_TRANSLATION_TOKEN` 

In [9]:
# Fix missing translations in staging.csv
def fix_missing_translations():
    """
    Loop through staging.csv and retry translation for any rows where translation equals MISSING_TRANSLATION_TOKEN
    """
    if not Path(STAGING_CSV_PATH).exists():
        print("❌ No staging.csv found")
        return
    
    # Load staging data
    staging_df = pd.read_csv(STAGING_CSV_PATH)
    print(f"Loaded {len(staging_df)} samples from staging.csv")
    
    # Find rows with missing translations
    missing_mask = staging_df['translation'] == MISSING_TRANSLATION_TOKEN
    missing_count = missing_mask.sum()
    
    if missing_count == 0:
        print("✅ No missing translations found!")
        return
    
    print(f"Found {missing_count} missing translations to fix\n")
    
    # Track statistics
    fixed_count = 0
    still_missing_count = 0
    
    # Iterate through rows with missing translations
    for idx in staging_df[missing_mask].index:
        row = staging_df.loc[idx]
        language = row['language']
        sentence = row['sentence']
        
        print(f"[{idx+1}/{len(staging_df)}] Translating {language}: {sentence[:50]}...")
        
        # Attempt translation
        translation = translate(language, sentence)
        
        if translation and translation != MISSING_TRANSLATION_TOKEN:
            # Update the dataframe
            staging_df.at[idx, 'translation'] = translation
            fixed_count += 1
            print(f"  ✓ Success: {translation[:60]}")
        else:
            print(f"  ✗ Failed: Encountered Google Translate error")
            print(f"            At index {idx}")
            print(f"  Still have {len([i for i in staging_df[missing_mask].index if i >= idx])} to re-translate!!")
            break
        
        # Small delay between requests to avoid rate limiting
        time.sleep(3)
    
    # Save updated staging.csv
    staging_df.to_csv(STAGING_CSV_PATH, index=False)
    
    print(f"\n{'='*60}")
    print(f"RESULTS")
    print(f"{'='*60}")
    print(f"✓ Fixed: {fixed_count}")
    print(f"📊 Total missing translations remaining: {(staging_df['translation'] == MISSING_TRANSLATION_TOKEN).sum()}")
    print(f"💾 Saved updated staging.csv")

# Run the function
# Uncomment to execute:
fix_missing_translations()

Loaded 2454 samples from staging.csv
Found 186 missing translations to fix

[1730/2454] Translating Portuguese: A capa da revista mostrou um cachorrinho fofo....
Translation found: The magazine cover featured an adorable puppy.
  ✓ Success: The magazine cover featured an adorable puppy.
[1731/2454] Translating Portuguese: Este serviço deve ser oferecido em um espaço que g...
Translation found: This service should be offered in a space that guarantees confidentiality and discretion.
  ✓ Success: This service should be offered in a space that guarantees co
[1732/2454] Translating Portuguese: Um, menininha, pulos, corda, através, lote estacio...
Translation found: A little girl, jumping rope, across the parking lot.
  ✓ Success: A little girl, jumping rope, across the parking lot.
[1733/2454] Translating Portuguese: Eu quero dizer que vou ganhar o dinheiro novamente...
Translation found: I mean that I'm going to earn the money again.
  ✓ Success: I mean that I'm going to earn the money ag

## View Staging Data

In [23]:
# Load and display staging data
if Path(STAGING_CSV_PATH).exists():
    staging_df = pd.read_csv(STAGING_CSV_PATH)
    print(f"Staging CSV: {len(staging_df)} total samples")
    print(f"Languages: {staging_df['language'].nunique()}")
    print(f"\nSamples per language:")
    print(staging_df['language'].value_counts())
    print(f"\nPreview:")
    display(staging_df.head(10))
else:
    print("No staging.csv found yet")

Staging CSV: 480 total samples
Languages: 8

Samples per language:
language
Danish       60
Greek        60
Czech        60
Bulgarian    60
Afrikaans    60
Estonian     60
Persian      60
Welsh        60
Name: count, dtype: int64

Preview:


,date,language,iso,cv_code,espeak_code,lineage,family_0,family_1,family_2,sentence,translation,wave,sampling_rate,IPA
0,NaN,Danish,dan,da,da,"['Indo-European', 'Classical Indo-European', '...",Indo-European,Germanic,North Germanic,"de så ud, som om alle nationers flag vajede i ...",they looked as if the flags of all nations wer...,staging_da_000.mp3,32000,dʔi sɒ ʔuð sʔʌm ʔʌm alə naʃʔonʔʌs flaj ʋ?ɑjəðə...
1,NaN,Danish,dan,da,da,"['Indo-European', 'Classical Indo-European', '...",Indo-European,Germanic,North Germanic,Dette er altafgørende for at fremme iværksætte...,This is crucial to promoting entrepreneurship ...,staging_da_001.mp3,32000,dʔetə ɛɐ̯ alt?ɑwɡˌœʔʌnə fʌ a fʁ?amə ʔiʋεɐ̯ksεt...
2,NaN,Danish,dan,da,da,"['Indo-European', 'Classical Indo-European', '...",Indo-European,Germanic,North Germanic,Somatiske tilstande er en hyppig årsag til aku...,Somatic conditions are a frequent cause of acu...,staging_da_002.mp3,32000,sʔomatʔisɡə tʔelstanə ɛɐ̯ ʔen hʔypʔik ɒɒsaj tʔ...
3,NaN,Danish,dan,da,da,"['Indo-European', 'Classical Indo-European', '...",Indo-European,Germanic,North Germanic,I januar lovede vi et parlamentsvenligt forman...,"In January, we promised a parliament-friendly ...",staging_da_003.mp3,32000,i janʔuˌɑ loʋəðə ʋi et pˌɑlamεntsʋεnlˌʔit fɒɒm...
4,NaN,Danish,dan,da,da,"['Indo-European', 'Classical Indo-European', '...",Indo-European,Germanic,North Germanic,"Det er upraktisk, bureaukratisk og unødvendigt...","It is impractical, bureaucratic and unnecessar...",staging_da_004.mp3,32000,de ɛɐ̯ upʁ?ɑktʔisk bʔyʁˌʔokʁ?ɑtʔisɡ ʌ unˌœðʋεn...
5,NaN,Danish,dan,da,da,"['Indo-European', 'Classical Indo-European', '...",Indo-European,Germanic,North Germanic,Denne finansiering vil således ikke automatisk...,This funding will therefore not be automatical...,staging_da_005.mp3,32000,dεnə fʔinansjʔeʔeŋ ʋʔel sʔʌleðəs ʔekə ˌɑwtʔoma...
6,NaN,Danish,dan,da,da,"['Indo-European', 'Classical Indo-European', '...",Indo-European,Germanic,North Germanic,Årsagerne til flugten kan imidlertid også være...,"However, the reasons for flight can also be et...",staging_da_006.mp3,32000,ɒɒsajˌʌnə tʔel flɒɡdən kan ʔimʔiðlʔʌtˌʔið ʔʌsə...
7,NaN,Danish,dan,da,da,"['Indo-European', 'Classical Indo-European', '...",Indo-European,Germanic,North Germanic,Byens adelige jomfruer sang og førte bruden fr...,The city's noble maidens sang and led the brid...,staging_da_007.mp3,32000,bʔyəns aðəlˌiə jʔʌmfʁuˌʔʌ s?ɑŋ ʌ fʔœɐ̯də bɐ̯ʔu...
8,NaN,Danish,dan,da,da,"['Indo-European', 'Classical Indo-European', '...",Indo-European,Germanic,North Germanic,Til trods for kritikpunkterne er den generelle...,"Despite the criticisms, the general direction ...",staging_da_008.mp3,32000,tʔel tʁʔoðs fʌ kʁʔitʔikpɒŋtˌʔʌnə ɛɐ̯ dɛn ɡʔenə...
9,NaN,Danish,dan,da,da,"['Indo-European', 'Classical Indo-European', '...",Indo-European,Germanic,North Germanic,den skønneste og uskyldigste pige er Anastasia...,The most beautiful and innocent girl is Anasta...,staging_da_009.mp3,32000,dɛn skʔœnəsdə ʌ ʔusɡyldisdə pˌiə ɛɐ̯ anastˌasʔ...


## Intelligently Add Samples to game_data.csv

This section intelligently selects samples from staging.csv and adds them to game_data.csv:

**Key features:**
1. **Avoids consecutive same-language days** - Each day will have a different language than the previous day (when possible)
2. **Sequential date assignment** - Days are assigned starting from the next available date in game_data.csv
3. **Audio file renaming** - Renames from `staging_XX_NNN.mp3` to `YYYY-MM-DD.mp3` format
4. **Automatic cleanup** - Removes used samples from staging.csv to prevent reuse

**How it works:**
- For each day, randomly selects from languages that are different from the previous day
- Only repeats a language consecutively if no other languages are available
- Tracks which samples have been used and removes them from staging

In [14]:
def add_staged_samples_to_game_data_intelligent(num_days_to_add=10):
    """
    Intelligently add N samples from staging.csv to game_data.csv:
    - Avoids consecutive days with the same language
    - Assigns consecutive dates starting from the next available date
    - Renames audio files from staging_XX_NNN.mp3 to YYYY-MM-DD.mp3
    - Removes used samples from staging.csv
    """
    if not Path(STAGING_CSV_PATH).exists():
        print("❌ No staging.csv found")
        return
    
    # Load staging data
    staging_df = pd.read_csv(STAGING_CSV_PATH)
    print(f"Loaded {len(staging_df)} samples from staging.csv")
    print(f"Languages available: {staging_df['language'].nunique()}")
    
    if len(staging_df) < num_days_to_add:
        print(f"⚠ Only {len(staging_df)} samples available, but you requested {num_days_to_add}")
        num_days_to_add = len(staging_df)
    
    # Load or create game_data
    if Path(GAME_DATA_PATH).exists():
        game_df = pd.read_csv(GAME_DATA_PATH, parse_dates=['date'])
        game_df['date'] = game_df['date'].dt.date
        next_date = max(game_df['date']) + timedelta(days=1)
        last_language = game_df.iloc[-1]['language']
        print(f"Loaded existing game_data.csv with {len(game_df)} entries")
        print(f"Next available date: {next_date}")
        print(f"Last language used: {last_language}")
    else:
        game_df = pd.DataFrame()
        next_date = date(2025, 6, 13)  # Your start date from batch_generator
        last_language = None
        print(f"Creating new game_data.csv starting from {next_date}")
    
    # Intelligent selection: avoid consecutive same-language days
    selected_indices = []
    staging_available = staging_df.copy()
    
    for day_idx in range(num_days_to_add):
        # Filter out the last language to avoid repetition
        if last_language is not None:
            available_candidates = staging_available[staging_available['language'] != last_language]
            
            # If all remaining samples are the same language as last, we must take it
            if len(available_candidates) == 0:
                print(f"  ⚠ Day {day_idx+1}: No other languages available, must repeat {last_language}")
                available_candidates = staging_available
        else:
            available_candidates = staging_available
        
        if len(available_candidates) == 0:
            print(f"  ❌ Ran out of samples after {day_idx} days")
            break
        
        # Randomly select one sample from available candidates
        selected_idx = available_candidates.sample(n=1, random_state=random.randint(0, 2**32-1)).index[0]
        selected_row = staging_available.loc[selected_idx]
        
        # Track selection
        selected_indices.append(selected_idx)
        last_language = selected_row['language']
        
        # Remove from available pool
        staging_available = staging_available.drop(selected_idx)
        
        print(f"  Day {day_idx+1}/{num_days_to_add}: Selected {selected_row['language']}")
    
    # Get the selected samples in order
    samples_to_add = staging_df.loc[selected_indices].copy().reset_index(drop=True)
    
    # Assign dates and rename audio files
    print(f"\n🔄 Assigning dates and renaming audio files...")
    for idx, row in samples_to_add.iterrows():
        current_date = next_date + timedelta(days=idx)
        samples_to_add.at[idx, 'date'] = current_date
        
        # Rename audio file from staging name to final date-based name
        old_audio_name = row['wave']
        new_audio_name = f"{current_date:%Y-%m-%d}.mp3"
        
        old_audio_path = AUDIO_DIR / old_audio_name
        new_audio_path = AUDIO_DIR / new_audio_name
        
        if old_audio_path.exists():
            shutil.move(str(old_audio_path), str(new_audio_path))
            samples_to_add.at[idx, 'wave'] = new_audio_name
            print(f"  ✓ {old_audio_name} → {new_audio_name}")
        else:
            print(f"  ⚠ Audio file not found: {old_audio_path}")
    
    # Append to game_data
    game_df = pd.concat([game_df, samples_to_add], ignore_index=True)
    
    # Save game_data
    game_df.to_csv(GAME_DATA_PATH, index=False)
    print(f"\n✅ Added {len(samples_to_add)} samples to game_data.csv")
    print(f"📊 Total game entries: {len(game_df)}")
    print(f"📅 Date range: {min(game_df['date'])} to {max(game_df['date'])}")
    
    # Remove used samples from staging
    remaining_staging = staging_df.drop(selected_indices)
    remaining_staging.to_csv(STAGING_CSV_PATH, index=False)
    print(f"\n🗑️  Removed {len(selected_indices)} used samples from staging.csv")
    print(f"📊 Remaining in staging: {len(remaining_staging)} samples")
    print(f"📊 Languages still available: {remaining_staging['language'].nunique()}")

# Example: Add 10 days worth of samples intelligently
# Uncomment to run:
add_staged_samples_to_game_data_intelligent(num_days_to_add=438)

Loaded 438 samples from staging.csv
Languages available: 41
Loaded existing game_data.csv with 2206 entries
Next available date: 2031-06-28
Last language used: Mandarin
  Day 1/438: Selected Persian
  Day 2/438: Selected Afrikaans
  Day 3/438: Selected Irish
  Day 4/438: Selected Nepali
  Day 5/438: Selected French
  Day 6/438: Selected Spanish
  Day 7/438: Selected Italian
  Day 8/438: Selected Macedonian
  Day 9/438: Selected Vietnamese
  Day 10/438: Selected Armenian
  Day 11/438: Selected Romanian
  Day 12/438: Selected Serbian
  Day 13/438: Selected Estonian
  Day 14/438: Selected Malayalam
  Day 15/438: Selected Russian
  Day 16/438: Selected Malayalam
  Day 17/438: Selected Hindi
  Day 18/438: Selected Slovak
  Day 19/438: Selected Malayalam
  Day 20/438: Selected Armenian
  Day 21/438: Selected Swahili
  Day 22/438: Selected Cantonese
  Day 23/438: Selected Malayalam
  Day 24/438: Selected Estonian
  Day 25/438: Selected Malayalam
  Day 26/438: Selected Nepali
  Day 27/438: Sel

/var/folders/yt/068fp3bx6rx1v421g4kkc62m0000gn/T/ipykernel_39205/2407162586.py:77: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2031-06-28' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  samples_to_add.at[idx, 'date'] = current_date


## Summary Stats

In [15]:
print("="*60)
print("SUMMARY")
print("="*60)

# Staging stats
if Path(STAGING_CSV_PATH).exists():
    staging_df = pd.read_csv(STAGING_CSV_PATH)
    print(f"\n📦 STAGING.CSV")
    print(f"  Total samples: {len(staging_df)}")
    print(f"  Languages: {staging_df['language'].nunique()}")
    print(f"  Ready to add to game_data.csv")
else:
    print(f"\n📦 STAGING.CSV: Not created yet")

# Game data stats
if Path(GAME_DATA_PATH).exists():
    game_df = pd.read_csv(GAME_DATA_PATH, parse_dates=['date'])
    game_df['date'] = game_df['date'].dt.date
    print(f"\n🎮 GAME_DATA.CSV")
    print(f"  Total entries: {len(game_df)}")
    print(f"  Date range: {min(game_df['date'])} to {max(game_df['date'])}")
    print(f"  Languages: {game_df['language'].nunique()}")
else:
    print(f"\n🎮 GAME_DATA.CSV: Not created yet")

# CV folders
print(f"\n📁 CV FOLDERS ({CV_BASE_DIR})")
available = scan_cv_directory(CV_BASE_DIR)
print(f"  Available to process: {len(available)} language(s)")

SUMMARY

📦 STAGING.CSV
  Total samples: 0
  Languages: 0
  Ready to add to game_data.csv

🎮 GAME_DATA.CSV
  Total entries: 2644
  Date range: 2025-06-13 to 2032-09-07
  Languages: 43

📁 CV FOLDERS (/Users/raymondtana/Desktop/cv)
  Available to process: 0 language(s)
